# N=100 saved-parameter bound-aware L-BFGS refinement

## Mandatory contract for users and AI agents

**Every run notebook must use this boilerplate design wherever possible. This notebook is the source of truth.** Copy its cell order, headings, guards, and adapter calls. Do not add, remove, reorder, merge, or redesign sections unless the user gives explicit permission to change the format in the current conversation. A request to create a run is not permission to change this style.

Code cells must remain declarative and personally editable by the user: change only the experiment prose and the exposed values (`run_name`, activation flags, parameters, runtime, initialization query, resources, database filters, sweep selections, and plot/output settings). Do not add notebook-local functions or classes, bespoke imports, loops, conditionals, subprocess or Slurm mechanics, database code, filesystem code, validation logic, device detection, batching logic, or plotting helpers. Reusable behavior belongs in `ofc.notebook_workflow` or the relevant package module, with only its arguments exposed here.

If this structure genuinely cannot represent a requested notebook, an agent must explain why and obtain explicit permission **before** deviating. See `scripts/AGENTS.md` for the scoped agent rules.

The config, direct-GPU, and Slurm sections are independent and guarded by first-line `Activated=False` flags. Leave unused sections inactive. Once data exists, a fresh kernel needs only setup, query, and the desired plot cells. The query can inherit the immutable YAML or select historical database rows. Use `max_cases_per_batch` to shard parameter cases and `max_initialisations_per_batch` to reduce per-device member memory, while keeping enough work per task that compilation and database writes do not dominate.

In [ ]:
from ofc.notebook_workflow import RunNotebook

run_name = "N100_three_cap_saved_parameter_lbfgs_refinement"
workflow = RunNotebook(run_name)

## Create the immutable config

Edit the arguments, then activate once. Returning later: leave `Activated=False`; the existing YAML is loaded without being changed. When `initialization_query` selects an old run, explicitly set `resume_optimizer: false` to reset Adam at the stored controls, or `resume_optimizer: true` to restore the matching stored Adam count and moments. Resuming requires exact, unperturbed `best` or `final` controls.

In [ ]:
Activated = False

description = "Bound-aware L-BFGS refinement of the top five saved controls for every distinct N=100 cap, smoothness, and sharpness combination."
reuse_existing = False
parameters = {
    "N": 100,
    "t_interval": 4.0,
    "r_bg": -0.008716,
    "u_isbound": True,
    "v_isbound": True,
    "u_max": [1280.0, 160.0, 40.0],
    "v_max": 1000.0,
    "slew_limit": 0.05,
    "optimizer": "lbfgs",  # Bound-aware L-BFGS-B in normalized controls.
    "schedule": [(50, 1.0)],
    "adam_learning_rate": 1e-2,
    "adam_beta1": 0.9,
    "adam_beta2": 0.999,
    "adam_eps": 1e-8,
    "lbfgs_history_size": 100,
    "lbfgs_max_linesearch_steps": 200,
    "lbfgs_tolerance": 1e-12,
    "smoothness": 0.0,  # Replaced by every exact value discovered at launch.
    "u_smooth": None,
    "v_smooth": None,
    "sharpness": 0.0,
    "u_sharp": None,
    "v_sharp": None,
    "block_size": 10,
    "J_tol": 1e-5,
    "u_tol": 1e-4,
    "v_tol": 1e-4,
    "projected_gradient_tol": 1e-5,
    "projected_gradient_alpha": 1.0,
}
runtime = {
    "initialisations": 0,
    "fourier_num_modes": 5,
    "fourier_rms_amplitude": 0.3,
    "fourier_intensity_fraction": 0.3,
    "use_jit": True,
    "use_x64": True,
    "device": "gpu",
    "concurrent_workers": 1,
    "max_cases_per_batch": None,  # None = one task per compile shape; or a positive case shard size.
    "max_initialisations_per_batch": None,  # None = all starts together; or a positive member shard size.
    "max_batch_elapsed_seconds": None,  # None = no per-batch limit; otherwise stop and record that batch after this many seconds.
    "max_elapsed_seconds": 12 * 60 * 60,  # Twelve hours for all cap batches together.
    "distribute_max_elapsed_across_batches": True,  # Three cap batches receive four hours each.
    "repeat_schedule_until_stable": True,
    "auto_halt": True,  # Stop once every member passes all enabled tolerances for 3 consecutive blocks.
    "database": "results/results.sqlite3",
}
initialization_query = {
    "where": {"status": "complete", "N": 100, "u_max": [1280.0, 160.0, 40.0]},
    "limit": 5,
    "order_by": "best_score",
    "descending": True,
    "control_kind": "best",
    "resume_optimizer": False,  # Fresh bound-aware L-BFGS state.
    "perturbed": False,
    "perturbation_levels": [0.0005, 0.001, 0.0025, 0.005, 0.01],
    "match_parameters": ["u_max", "smoothness", "sharpness"],
    "discover_parameters": ["u_max", "smoothness", "sharpness"],
    "discover_group_parameters": ["u_max"],
}

config_document = workflow.create_config(
    activated=Activated,
    description=description,
    parameters=parameters,
    runtime=runtime,
    initialization_query=initialization_query,
    reuse_existing=reuse_existing,
)

## Run directly on `bar`'s GPU (detached)

This launches outside Slurm, verifies that JAX can see the GPU, and detaches the process into its own session and log so it survives a browser or laptop disconnect. The immutable config must use `device: auto` or `device: gpu`.

In [ ]:
Activated = False

queue_id = 26214844451112451
python_executable = None  # None uses this kernel's Python; otherwise give a path.
extra_arguments = []  # Example: ["--batch-index", "0"].
detached = True  # Keep running if the notebook/browser disconnects.
log_path = "logs/N100_three_cap_saved_parameter_lbfgs_refinement.log"

active_queue_id = workflow.run_on_bar_gpu(
    activated=Activated,
    queue_id=queue_id,
    python_executable=python_executable,
    extra_arguments=extra_arguments,
    detached=detached,
    log_path=log_path,
)

## Submit through Slurm (alternative)

In [ ]:
Activated = False

partition = "zen5,epyc"
time = "4-03:00:00"
cpus = 32
memory = "64G"
array = None  # None = array when multiple config batches; True or False overrides.
array_max_concurrent = None  # Example: 4; None leaves the array unthrottled.
job_name = None  # None uses run_name.
extra_arguments = []  # Extra sbatch arguments, e.g. ["--constraint=..."].

active_queue_id = workflow.submit_slurm(
    activated=Activated,
    partition=partition,
    time=time,
    cpus=cpus,
    memory=memory,
    array=array,
    array_max_concurrent=array_max_concurrent,
    job_name=job_name,
    extra_arguments=extra_arguments,
)

## Query persisted data

This is read-only and does not depend on executing any optional cell above. It supports any number of varying parameters; choose the dimensions used by each plot later. With config inheritance enabled, the existing YAML automatically supplies its database and immutable config identity (and the selected rows carry all resolved config parameters). Disable inheritance to query older runs using only database filters.

In [ ]:
inherit_config = True  # True = use run_config/<run_name>.yaml automatically; False = historical database only.
database = None  # None = inherited config database, or results/results.sqlite3 when inheritance is False.
queue_id = None  # None = latest matching execution; or an integer such as 700843.
config_run_rank = 1  # 1 = latest, 2 = second latest, etc.
statuses = None  # None = all; or "running", "complete", "failed", or a list.
# With inherit_config=False, identify old data here, e.g. {"config_name": "old_run"}.
filters = {}  # Exact: {"N": 100}; list: {"u_max": [40, 160]}; range: {"best_score": (0, 100)}.
# Any number of dimensions. None discovers all varying parameters automatically.
# Names: N, t_interval, r_bg, u_isbound, v_isbound, u_max, v_max, slew_limit,
# optimizer, schedule, adam_learning_rate, adam_beta1, adam_beta2, adam_eps,
# lbfgs_history_size, lbfgs_max_linesearch_steps, lbfgs_tolerance, smoothness,
# u_smooth, v_smooth, sharpness, u_sharp, v_sharp, block_size, J_tol, u_tol, v_tol,
# projected_gradient_tol, projected_gradient_alpha.
sweep_parameters = ["u_max", "smoothness", "sharpness"]
require_saved_stage = True  # True excludes registered runs with no saved stage yet.
limit = None  # None = unlimited; otherwise a positive integer.
order_by = "run_id"  # Examples: "run_id", "best_score", "u_max", "started_utc".
descending = False

query_result = workflow.query(
    inherit_config=inherit_config,
    database=database,
    queue_id=queue_id,
    config_run_rank=config_run_rank,
    statuses=statuses,
    filters=filters,
    sweep_parameters=sweep_parameters,
    require_saved_stage=require_saved_stage,
    limit=limit,
    order_by=order_by,
    descending=descending,
)

## Figure display and saving

In [ ]:
save_figure = None  # None = display only; "/" = figures/; or "/experiment/latest".
figure_format = "png"  # "png" or "pdf".
preview_dpi = 180  # Increase this if inline multi-sweep labels are hard to read.
save_dpi = 600  # Resolution used when saving PNG figures.

## Unified sweep summary

The history median/spread and objective strip use only the runs shown in each row. Dashed vertical lines mark learning-rate changes.


In [ ]:
sweep_parameter = None  # Required when the query selected multiple sweeps; e.g. "u_max".
history_points = 1200
summary_figure = query_result.plot_summary(
    sweep_parameter=sweep_parameter,
    history_points=history_points,
)
workflow.present_figure(
    summary_figure,
    "01_sweep_summary",
    save_figure=save_figure,
    figure_format=figure_format,
    preview_dpi=preview_dpi,
    save_dpi=save_dpi,
)


## Double sweep summary

In [ ]:
separate_sweep_parameter = "u_max"  # One rectangle per value.
colour_sweep_parameter = "sharpness"  # Coloured best trace/control per value.
history_points = 1200
double_sweep_figure = query_result.plot_double_sweep_summary(
    separate_sweep_parameter=separate_sweep_parameter,
    colour_sweep_parameter=colour_sweep_parameter,
    history_points=history_points,
)
workflow.present_figure(
    double_sweep_figure,
    "02_double_sweep_summary",
    save_figure=save_figure,
    figure_format=figure_format,
    preview_dpi=preview_dpi,
    save_dpi=save_dpi,
)

## Triple sweep summary

In [ ]:
row_sweep_parameter = "u_max"
column_sweep_parameter = "smoothness"
colour_sweep_parameter = "sharpness"
history_points = 1200
triple_sweep_figure = query_result.plot_triple_sweep_summary(
    row_sweep_parameter=row_sweep_parameter,
    column_sweep_parameter=column_sweep_parameter,
    colour_sweep_parameter=colour_sweep_parameter,
    history_points=history_points,
)
workflow.present_figure(
    triple_sweep_figure,
    "03_triple_sweep_summary",
    save_figure=save_figure,
    figure_format=figure_format,
    preview_dpi=preview_dpi,
    save_dpi=save_dpi,
)